# New

In [16]:
import pandas as pd
df = pd.read_csv("data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
df["label"].value_counts()

label
2 (Jazz)                    1444
1 (Hip-Hop + Pop + HJDB)    1426
3 (Rock + Pop)              1269
4 (Classical)               1054
1 + 3                        162
2 + 3                        126
2 + 4                         75
Name: count, dtype: int64

In [ ]:
df_gtzan = df[df["file"].str.startswith("gtzan")]   # sanity check thtat gtzan belongs to 1 cluster
df_gtzan["label"].unique()

array(['3 (Rock + Pop)', '2 (Jazz)', '4 (Classical)',
       '1 (Hip-Hop + Pop + HJDB)'], dtype=object)

In [47]:
import pandas as pd
df = pd.read_csv("data_cluster_assignments/df_hard_6_2clusters_new.csv")
print(df["label"].value_counts())
df_gtzan = df[df["file"].str.startswith("gtzan")]   # sanity check thtat gtzan belongs to 1 cluster
print(df_gtzan["label"].unique())

label
1 (Pop + Groove + Rock + Hip-Hop)    2993
2 (Classical + Jazz)                 2563
Name: count, dtype: int64
['1 (Pop + Groove + Rock + Hip-Hop)' '2 (Classical + Jazz)']


In [11]:
import pandas as pd
df = pd.read_csv("data_cluster_assignments/df_raw_113_3clusters_new.csv")
# some weird formatting for some files: some of them are of the form file/track, that is why we remove the second part
df["file"] = df["file"].apply(lambda x: x.split("/")[0])
rwc_files = [file for file in df["file"].values if "rwc" in file]
df["label"].value_counts()
#df.to_csv("data_cluster_assignments/df_raw_113_3clusters_new.csv")

label
3 (Jazz + Pop + Rock)              1912
1 (Classical + Groove + Jazz)      1822
2 (Pop + Rock + HJDB + Hip-Hop)    1822
Name: count, dtype: int64

In [12]:
import numpy as np
import os
import shutil
from tqdm import tqdm
# cluster number starts with 1 here!
def prepare_cluster_data(cluster_number, clustering_configuration, SAVE_NPZ, df_path):
    root = "/hpcwork/ui556004/data/beat_this/"
    root_save = os.path.join(root, "clustering_configurations", clustering_configuration)
    save_spectrograms_path = os.path.join(root_save,  f"cluster_{cluster_number}/data/audio/spectrograms")
    os.makedirs(save_spectrograms_path, exist_ok = True)
    os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
    annotations_path = os.path.join(root_save, f"cluster_{cluster_number}/data/annotations")

    shutil.copytree("/hpcwork/ui556004/data/beat_this/annotations", annotations_path, dirs_exist_ok = True)
    df = pd.read_csv(df_path)
    df_filtered = df[df["label"].str.contains(str(cluster_number))]
    files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used = set([file.split("___")[0] for file in files])
    print(len(df_filtered))
    print(datasets_used)

    gtzan_files = 0
    for dataset in tqdm(datasets_used):
        dataset_files = {}
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
        print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        for piece in selected_files:
            # getting all augmentations of the same file
            all_similar = [file for file in lst if piece in file ]
            if (len(all_similar) != 22 and len(all_similar) == 1):
                gtzan_files += 1
                #print(len(all_similar))
            
            pieces = {f"{file}" : data_npz[file] for file in all_similar}
            dataset_files = {**dataset_files, **pieces }
            #cluster = {**cluster, **pieces }
        
        path_to_save = os.path.join(save_spectrograms_path , f"{dataset}.npz")
        if SAVE_NPZ:
            np.savez(path_to_save, **dataset_files)
            print(f"saved npz of {dataset}")
        #datasets_npz[dataset] =dataset_files
    # sanity check: except for gtzan, each file should be repeated 22 times
    #assert (len(cluster) - gtzan_files) / 22 + gtzan_files == len(df_filtered)

In [13]:
for cluster_number in range(1,4):
    print(f"preparing data for cluster {cluster_number}")
    prepare_cluster_data(cluster_number = cluster_number, clustering_configuration ="clusters3_layer113" , SAVE_NPZ = True, df_path = "data_cluster_assignments/df_raw_113_3clusters_new.csv")

preparing data for cluster 1
1822
{'hainsworth', 'candombe', 'beatles', 'simac', 'smc', 'ballroom', 'rwc', 'hjdb', 'harmonix', 'asap', 'gtzan', 'groove_midi', 'filosax', 'guitarset', 'jaah', 'tapcorrect'}


  0%|          | 0/16 [00:00<?, ?it/s]

number of selected files from the hainsworth is 51


  6%|▋         | 1/16 [00:05<01:22,  5.52s/it]

saved npz of hainsworth
number of selected files from the candombe is 17


...

saved npz of guitarset
number of selected files from the jaah is 50


 93%|█████████▎| 14/15 [03:18<00:10, 10.37s/it]

saved npz of jaah
number of selected files from the tapcorrect is 45


100%|██████████| 15/15 [03:33<00:00, 14.21s/it]

saved npz of tapcorrect


In [49]:
for cluster_number in range(1,3):
    print(f"preparing data for cluster {cluster_number}")
    prepare_cluster_data(cluster_number = cluster_number, clustering_configuration ="clusters2_layer6" , SAVE_NPZ = True, df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv")

preparing data for cluster 1
2993
{'smc', 'gtzan', 'simac', 'tapcorrect', 'beatles', 'harmonix', 'filosax', 'hjdb', 'candombe', 'rwc', 'groove_midi', 'hainsworth', 'jaah', 'ballroom'}


  0%|          | 0/14 [00:00<?, ?it/s]

number of selected files from the smc is 17


  7%|▋         | 1/14 [00:01<00:19,  1.49s/it]

saved npz of smc
number of selected files from the gtzan is 652


...

saved npz of hainsworth
number of selected files from the jaah is 106


 93%|█████████▎| 13/14 [08:08<00:24, 24.93s/it]

saved npz of jaah
number of selected files from the ballroom is 286


100%|██████████| 14/14 [08:36<00:00, 36.87s/it]

saved npz of ballroom


In [18]:
for cluster_number in range(1,5):
    print(f"preparing data for cluster {cluster_number}")
    prepare_cluster_data(cluster_number = cluster_number, clustering_configuration ="clusters4_layer12" , SAVE_NPZ = True, df_path = "data_cluster_assignments/df_cmeans_12_4clusters_new.csv")

preparing data for cluster 1
1588
{'smc', 'gtzan', 'simac', 'tapcorrect', 'beatles', 'harmonix', 'filosax', 'hjdb', 'candombe', 'rwc', 'groove_midi', 'hainsworth', 'jaah', 'ballroom'}


  0%|          | 0/14 [00:00<?, ?it/s]

number of selected files from the smc is 16


  7%|▋         | 1/14 [00:01<00:16,  1.27s/it]

saved npz of smc
number of selected files from the gtzan is 238


...

saved npz of hainsworth
number of selected files from the jaah is 3


 93%|█████████▎| 14/15 [04:42<00:05,  5.71s/it]

saved npz of jaah
number of selected files from the ballroom is 105


100%|██████████| 15/15 [04:51<00:00, 19.45s/it]

saved npz of ballroom


In [32]:
prepare_cluster_data(cluster_number = 1, clustering_configuration ="clusters4_layer12" , SAVE_NPZ = True, df_path = "data_cluster_assignments/df_cmeans_12_4clusters.csv")

1160
{'groove_midi', 'asap', 'beatles', 'jaah', 'smc', 'gtzan', 'guitarset', 'rwc', 'ballroom', 'harmonix', 'tapcorrect', 'hainsworth', 'simac'}


  0%|          | 0/13 [00:00<?, ?it/s]

number of selected files from the groove_midi is 2


  8%|▊         | 1/13 [00:00<00:04,  2.92it/s]

saved npz of groove_midi
number of selected files from the asap is 473


...

saved npz of tapcorrect
number of selected files from the hainsworth is 56


 92%|█████████▏| 12/13 [03:42<00:06,  6.49s/it]

saved npz of hainsworth
number of selected files from the simac is 122


100%|██████████| 13/13 [03:50<00:00, 17.70s/it]

saved npz of simac


## analysing the data tables

In [41]:
import pandas as pd 
df = pd.read_csv("data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
df["label"].value_counts()

label
2 (Jazz)                    1444
1 (Hip-Hop + Pop + HJDB)    1426
3 (Rock + Pop)              1269
4 (Classical)               1054
1 + 3                        162
2 + 3                        126
2 + 4                         75
Name: count, dtype: int64

In [110]:
import pandas as pd 
df = pd.read_csv("data_cluster_assignments/df_hard_6_2clusters_new.csv")
df["label"].value_counts()

label
1 (Pop + Groove + Rock + Hip-Hop)    2993
2 (Classical + Jazz)                 2563
Name: count, dtype: int64

In [15]:
import pandas as pd 
df = pd.read_csv("data_cluster_assignments/df_raw_113_3clusters_new.csv")
df["label"].value_counts()

label
3 (Jazz + Pop + Rock)              1912
1 (Classical + Groove + Jazz)      1822
2 (Pop + Rock + HJDB + Hip-Hop)    1822
Name: count, dtype: int64

In [6]:
# some weird formatting for some files: some of them are of the form file/track, that is why we remove the second part
df["file"] = df["file"].apply(lambda x: x.split("/")[0])
rwc_files = [file for file in df["file"].values if "rwc" in file]
#rwc_files

In [136]:
import numpy as np  #sanity check
cluster_number = 1
data = np.load(os.path.join("/hpcwork/ui556004/data/beat_this/clustering_configurations/clusters3_layer113", f"cluster_{cluster_number}", "data/audio/spectrograms", f"jaah.npz"))
lst = data.files
print(len(lst))

1122


In [2]:
import numpy as np
import os
from tqdm import tqdm
import pandas as pd
import shutil
root = "/hpcwork/ui556004/data/beat_this/"
def get_split_files(df, cluster_number):
    
    # root_save = "/hpcwork/ui556004/data/beat_this/clusters"
    # save_spectrograms_path = os.path.join(root_save, f"cluster_{cluster_number}/data/audio/spectrograms")
    # annotations_path = os.path.join(root_save, f"cluster_{cluster_number}/data/annotations")
    cluster = {}
    df_filtered = df[df["label"].str.contains(str(cluster_number))]
    files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used = set([file.split("___")[0] for file in files])
    train_val_split = {}
    for dataset in tqdm(datasets_used):
        validation_files_new = 0
        train_files_new = 0
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
        #print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        if dataset != "gtzan":
            split = pd.read_csv(f"/hpcwork/ui556004/data/beat_this/data/annotations/{dataset}/single.split", sep = "\t", header = None, names = ["File", "Split"])
            split_filtered = split[split["Split"] != "train"]
            val_files_split = list(split_filtered["File"].values)
            for piece in selected_files:
                if piece in val_files_split:
                    validation_files_new += 1
                else:
                    train_files_new += 1
            train_val_split[dataset] =  (validation_files_new, train_files_new)
    data_npz = np.load(os.path.join("/hpcwork/ui556004/data/beat_this/clustering_configurations/clusters3_layer113", f"cluster_{cluster_number}", "data/audio/spectrograms", f"gtzan.npz"))
    lst = data_npz.files
    test_len = len(lst)
        
    return train_val_split, test_len, lst

In [3]:
def get_split_percentage(train_val_split):
    validation_items = 0 #4
    train_items = 0
    for keys, values in train_val_split.items():
        validation_items += values[0]
        train_items += values[1]
    print(validation_items)
    print(train_items)
    print(f"{validation_items / train_items * 100}%")
    return validation_items, train_items

In [52]:
all_labels = [label for label in list(df["label"].unique()) if "(" in label]
all_labels

['1 (Classical + Groove + Jazz)',
 '3 (Jazz + Pop + Rock)',
 '2 (Pop + Rock + HJDB + Hip-Hop)']

In [53]:
df["label"].unique()

array(['1 (Classical + Groove + Jazz)', '3 (Jazz + Pop + Rock)',
       '2 (Pop + Rock + HJDB + Hip-Hop)'], dtype=object)

In [16]:
total_train = 0
total_val = 0
total_test = 0
test_files_clusters = []
all_labels = {int(label[0]): label[1:] for label in list(df["label"].unique()) if "(" in label}
for cluster_number in range(1,4):
    
    train_val_split, test_len, test_files = get_split_files(df, cluster_number)
    test_files_clusters += test_files
    print(f"results for the cluster {cluster_number} ({all_labels[cluster_number]})")
    validation_items, train_items =get_split_percentage(train_val_split)
    total_train += train_items
    total_val += validation_items
    total_test += test_len
    print(f"total number of files cluster {cluster_number} is {validation_items + train_items + test_len} ")
print("overall results:")
print(total_val)
print(total_train)
print(total_val / total_train * 100)
print(f"total files {total_val + total_train + total_test}")

100%|██████████| 16/16 [00:01<00:00,  8.76it/s]


results for the cluster 1 ( (Classical + Groove + Jazz))
184
1434
12.831241283124129%
total number of files cluster 1 is 1822 


100%|██████████| 12/12 [00:00<00:00, 25.27it/s]


results for the cluster 2 ( (Pop + Rock + HJDB + Hip-Hop))
204
1212
16.831683168316832%
total number of files cluster 2 is 1822 


100%|██████████| 15/15 [00:00<00:00, 26.67it/s]

results for the cluster 3 ( (Jazz + Pop + Rock))
168
1355
12.398523985239853%
total number of files cluster 3 is 1912 
overall results:
556
4001
13.896525868532866
total files 5556


In [ ]:
gtzan_files = [file.split("___")[1] for file in df["file"].values if "gtzan" in file]
test_files_clusters = [file.split("/")[0] for file in test_files_clusters if "gtzan" in file]
set(gtzan_files).difference(set(test_files_clusters))

In [45]:
test_files_cluster1 = set(test_files_cluster[1])
test_files_cluster2 = set(test_files_cluster[2])
test_files_cluster3 = set(test_files_cluster[3])
len(test_files_cluster1) + len(test_files_cluster2) + len(test_files_cluster3)

894

In [53]:
total_train = 0
total_val = 0
all_labels = {int(label[0]): label[1:] for label in list(df["label"].unique()) if "(" in label}
for cluster_number in range(1,3):
    
    train_val_split, test_len = get_split_files(df, cluster_number)
    print(f"results for the cluster {cluster_number} ({all_labels[cluster_number]})")
    validation_items, train_items =get_split_percentage(train_val_split)
    total_train += train_items
    total_val += validation_items
    print(f"total number of files cluster {cluster_number} is {validation_items + train_items + test_len} ")
print("overall results:")
print(total_val)
print(total_train)
print(total_val / total_train * 100)
print(f"total files {total_val + total_train + 993}")

100%|██████████| 14/14 [00:00<00:00, 21.96it/s]


results for the cluster 1 ( (Pop + Groove + Rock + Hip-Hop))
347
1994
17.402206619859577%
total number of files cluster 1 is 2579 


100%|██████████| 14/14 [00:00<00:00, 27.25it/s]

results for the cluster 2 ( (Classical + Jazz))
209
2007
10.413552566018934%
total number of files cluster 2 is 2437 
overall results:
556
4001
13.896525868532866
total files 5550


In [42]:
total_train = 0
total_val = 0
all_labels = {int(label[0]): label[1:] for label in list(df["label"].unique()) if "(" in label}
for cluster_number in range(1,5):
    
    train_val_split, test_len = get_split_files(df, cluster_number)
    print(f"results for the cluster {cluster_number} ({all_labels[cluster_number]})")
    validation_items, train_items =get_split_percentage(train_val_split)
    total_train += train_items
    total_val += validation_items
    print(f"total number of files cluster {cluster_number} is {validation_items + train_items + test_len} ")
print("overall results:")
print(total_val)
print(total_train)
print(total_val / total_train * 100)
print(f"total files {total_val + total_train + 993}")

100%|██████████| 14/14 [00:01<00:00,  8.18it/s]


results for the cluster 1 ( (Hip-Hop + Pop + HJDB))
220
1130
19.469026548672566%
total number of files cluster 1 is 1588 


100%|██████████| 13/13 [00:00<00:00, 23.61it/s]


results for the cluster 2 ( (Jazz))
123
1301
9.454265949269793%
total number of files cluster 2 is 1645 


100%|██████████| 11/11 [00:00<00:00, 22.37it/s]


results for the cluster 3 ( (Rock + Pop))
153
969
15.789473684210526%
total number of files cluster 3 is 1557 


100%|██████████| 15/15 [00:00<00:00, 23.85it/s]


results for the cluster 4 ( (Classical))
106
918
11.546840958605664%
total number of files cluster 4 is 1129 
overall results:
602
4318
13.941639647985179
total files 5913


In [ ]:
# making sure that not all jazz validation files are in cluster 2 + 3 and 2 + 4
datasets = {'smc',  'simac', 'tapcorrect', 'guitarset', 'beatles', 'harmonix', 'filosax', 'hjdb', 'rwc', 'hainsworth', 'jaah', 'ballroom'}
list_23 = list(df[df["label"] == "2 + 3"]["file"].values)
list_24 = list(df[df["label"] == "2 + 4"]["file"].values)
val_files = []
for dataset in datasets:
    split = pd.read_csv(f"/hpcwork/ui556004/data/beat_this/data/annotations/{dataset}/single.split", sep = "\t", header = None, names = ["File", "Split"])
    split_filtered = split[split["Split"] != "train"]
    val_files_split = list(split_filtered["File"].values)
    
    intersection_23 = list(set(list_23) & set(val_files_split))
    intersection_24 = list(set(list_24) & set(val_files_split))
    val_files.extend(intersection_23)
    val_files.extend(intersection_24)
val_files

[]

# Old

In [1]:
import pandas as pd
df = pd.read_csv("sorted_files_beat_this_4.csv")
df["label"].value_counts()

label
1 (Pop + HJDB + Hip-Hop)    1371
3 (Jazz)                    1370
0 (Pop + Rock + Metal)      1249
2 (Classical)                969
2 + 3                        129
0 + 3                         70
0 + 1                         32
1 + 3                         30
Name: count, dtype: int64

In [ ]:
import numpy as np
import os
import shutil
from tqdm import tqdm
def prepare_cluster_data(cluster_number, clustering_configuration, SAVE_NPZ, df):
    root = "/hpcwork/ui556004/data/beat_this/"
    root_save = "/hpcwork/ui556004/data/beat_this/clustering_configurations"
    save_spectrograms_path = os.path.join(root_save, clustering_configuration,  f"cluster_{cluster_number}/data/audio/spectrograms")
    os.makedirs(save_spectrograms_path, exist_ok = True)
    os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
    annotations_path = os.path.join(root_save, f"cluster_{cluster_number}/data/annotations")

    shutil.copytree("/hpcwork/ui556004/data/beat_this/clusters/annotations", annotations_path, dirs_exist_ok = True)
    df_filtered = df[df["label"].str.contains(str(cluster_number))]
    files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used = set([file.split("___")[0] for file in files])
    print(len(df_filtered))
    print(datasets_used)

    gtzan_files = 0
    for dataset in tqdm(datasets_used):
        dataset_files = {}
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
        print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        for piece in selected_files:
            # getting all augmentations of the same file
            all_similar = [file for file in lst if piece in file ]
            if (len(all_similar) != 22 and len(all_similar) == 1):
                gtzan_files += 1
                #print(len(all_similar))
            
            pieces = {f"{file}" : data_npz[file] for file in all_similar}
            dataset_files = {**dataset_files, **pieces }
            #cluster = {**cluster, **pieces }
        
        path_to_save = os.path.join(save_spectrograms_path , f"{dataset}.npz")
        if SAVE_NPZ:
            np.savez(path_to_save, **dataset_files)
            print(f"saved npz of {dataset}")
        #datasets_npz[dataset] =dataset_files
    # sanity check: except for gtzan, each file should be repeated 22 times
    #assert (len(cluster) - gtzan_files) / 22 + gtzan_files == len(df_filtered)

In [ ]:
# creating small cluster for tests
import numpy as np
import os
import shutil
from tqdm import tqdm

cluster_number = 2
SAVE_NPZ = True
trying_set = True 

cluster_name = "try" if trying_set else cluster_number
root = "/hpcwork/ui556004/data/beat_this/"
root_save = "/hpcwork/ui556004/data/beat_this/clusters"
save_spectrograms_path = os.path.join(root_save, f"cluster_{cluster_name}/data/audio/spectrograms")
annotations_path = os.path.join(root_save, f"cluster_{cluster_name}/data/annotations")
os.makedirs(save_spectrograms_path, exist_ok = True)
shutil.copytree("/hpcwork/ui556004/data/beat_this/clusters/annotations", annotations_path, dirs_exist_ok = True)
os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
df_filtered = df[df["label"].str.contains(str(cluster_number))]
files = list(df_filtered["file"].values)
# getting all files and corresponding datasets
files_wo_dataset = [file.split("___")[1] for file in files]
datasets_used = set([file.split("___")[0] for file in files])
print(len(df_filtered))
print(datasets_used)
num_files = 5

gtzan_files = 0
for dataset in tqdm(datasets_used):
    if dataset != "gtzan":
        dataset_files = {}
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = [file.split("/")[0] for file in lst]
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset][:5]
        print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        for piece in selected_files:
            # getting all augmentations of the same file
            all_similar = [file for file in lst if piece in file ]
            if (len(all_similar) != 22 and len(all_similar) == 1):
                gtzan_files += 1
                #print(len(all_similar))
            
            pieces = {f"{file}" : data_npz[file] for file in all_similar}
            dataset_files = {**dataset_files, **pieces }
            #cluster = {**cluster, **pieces }
        
    path_to_save = os.path.join(save_spectrograms_path , f"{dataset}.npz")
    if SAVE_NPZ:
        np.savez(path_to_save, **dataset_files)
        print(f"saved npz of {dataset}")
        #datasets_npz[dataset] =dataset_files
    # sanity check: except for gtzan, each file should be repeated 22 times
    #assert (len(cluster) - gtzan_files) / 22 + gtzan_files == len(df_filtered)

1098
{'gtzan', 'hainsworth', 'smc', 'harmonix', 'simac', 'rwc', 'beatles', 'ballroom', 'jaah', 'filosax', 'tapcorrect', 'guitarset', 'asap'}


 15%|█▌        | 2/13 [00:00<00:00, 13.12it/s]

saved npz of gtzan
number of selected files from the hainsworth is 0
saved npz of hainsworth
number of selected files from the smc is 5
saved npz of smc


 38%|███▊      | 5/13 [00:00<00:01,  7.29it/s]

number of selected files from the harmonix is 0
saved npz of harmonix
number of selected files from the simac is 0
saved npz of simac
number of selected files from the rwc is 5


...

number of selected files from the ballroom is 0
saved npz of ballroom
number of selected files from the jaah is 0
saved npz of jaah
number of selected files from the filosax is 5


 92%|█████████▏| 12/13 [00:02<00:00,  4.19it/s]

saved npz of filosax
number of selected files from the tapcorrect is 0
saved npz of tapcorrect
number of selected files from the guitarset is 0
saved npz of guitarset
number of selected files from the asap is 5


100%|██████████| 13/13 [00:03<00:00,  3.96it/s]

saved npz of asap


In [3]:
prepare_cluster_data(cluster_number=3, SAVE_NPZ=True)

1599
{'smc', 'beatles', 'simac', 'hainsworth', 'filosax', 'rwc', 'gtzan', 'jaah', 'ballroom', 'harmonix', 'hjdb', 'guitarset', 'tapcorrect'}


  0%|          | 0/13 [00:00<?, ?it/s]

number of selected files from the smc is 57


  8%|▊         | 1/13 [00:05<01:06,  5.54s/it]

saved npz of smc
number of selected files from the beatles is 21


...

saved npz of hjdb
number of selected files from the guitarset is 176


 92%|█████████▏| 12/13 [02:55<00:13, 13.33s/it]

saved npz of guitarset
number of selected files from the tapcorrect is 26


100%|██████████| 13/13 [03:07<00:00, 14.44s/it]

saved npz of tapcorrect


In [3]:
prepare_cluster_data(cluster_number=1, SAVE_NPZ=True)

1433
{'candombe', 'hainsworth', 'hjdb', 'ballroom', 'smc', 'gtzan', 'harmonix', 'rwc', 'simac', 'tapcorrect', 'beatles', 'jaah'}


  0%|          | 0/12 [00:00<?, ?it/s]

number of selected files from the candombe is 35


  8%|▊         | 1/12 [00:08<01:33,  8.46s/it]

saved npz of candombe
number of selected files from the hainsworth is 41


...

saved npz of tapcorrect
number of selected files from the beatles is 5


 92%|█████████▏| 11/12 [03:27<00:14, 14.84s/it]

saved npz of beatles
number of selected files from the jaah is 4


100%|██████████| 12/12 [03:28<00:00, 17.39s/it]

saved npz of jaah


In [16]:
import numpy as np
import os
from tqdm import tqdm
import pandas as pd
import shutil
def get_split_files(cluster_number):
    root = "/hpcwork/ui556004/data/beat_this/"
    root_save = "/hpcwork/ui556004/data/beat_this/clusters"
    save_spectrograms_path = os.path.join(root_save, f"cluster_{cluster_number}/data/audio/spectrograms")
    annotations_path = os.path.join(root_save, f"cluster_{cluster_number}/data/annotations")
    os.makedirs(save_spectrograms_path, exist_ok = True)
    #shutil.copytree("/hpcwork/ui556004/data/beat_this/clusters/annotations", annotations_path, dirs_exist_ok = True)
    #os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
    # filtering files that belong to cluster number 0
    cluster = {}
    df_filtered = df[df["label"].str.contains(str(cluster_number))]
    files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used = set([file.split("___")[0] for file in files])
    train_val_split = {}
    for dataset in tqdm(datasets_used):
        validation_files_new = 0
        train_files_new = 0
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
        #print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        if dataset != "gtzan":
            split = pd.read_csv(f"/hpcwork/ui556004/data/beat_this/clusters/cluster_0/data/annotations/{dataset}/single.split", sep = "\t", header = None, names = ["File", "Split"])
            split_filtered = split[split["Split"] != "train"]
            val_files_split = list(split_filtered["File"].values)
            for piece in selected_files:
                if piece in val_files_split:
                    validation_files_new += 1
                else:
                    train_files_new += 1
            train_val_split[dataset] =  (validation_files_new, train_files_new)
        
    return train_val_split
    


In [17]:

root_save = "/hpcwork/ui556004/data/beat_this/clusters"
shutil.copytree("/hpcwork/ui556004/data/beat_this/clusters/annotations", os.path.join(root_save, f"cluster_{0}/data/annotations"), dirs_exist_ok = True)

'/hpcwork/ui556004/data/beat_this/clusters/cluster_0/data/annotations'

In [8]:
get_split_files(2)

100%|██████████| 13/13 [00:00<00:00, 19.26it/s]


{'tapcorrect': (2, 3),
 'ballroom': (16, 108),
 'rwc': (16, 67),
 'asap': (65, 408),
 'hainsworth': (10, 52),
 'simac': (0, 80),
 'guitarset': (0, 4),
 'filosax': (1, 7),
 'jaah': (1, 2),
 'harmonix': (0, 1),
 'smc': (0, 143),
 'beatles': (0, 1)}

In [1]:
def get_split_percentage(train_val_split):
    validation_items = 0 #4
    train_items = 0
    for keys, values in train_val_split.items():
        validation_items += values[0]
        train_items += values[1]
    print(validation_items)
    print(train_items)
    print(validation_items / train_items * 100)

In [20]:
for cluster_number in range(4):
    train_val_split = get_split_files(cluster_number)
    print(f"results for the cluster {cluster_number}")
    get_split_percentage(train_val_split)

100%|██████████| 12/12 [00:01<00:00, 11.32it/s]


results for the cluster 0
119
794
14.987405541561714


100%|██████████| 12/12 [00:00<00:00, 22.94it/s]


results for the cluster 1
179
958
18.684759916492695


100%|██████████| 13/13 [00:00<00:00, 20.76it/s]


results for the cluster 2
111
876
12.67123287671233


100%|██████████| 13/13 [00:00<00:00, 26.08it/s]

results for the cluster 3
112
1298
8.628659476117104


In [30]:
validation_items = 0 #4
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


112
731
15.321477428180575


In [28]:
validation_items = 0  #3
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


147
781
18.82202304737516


In [ ]:
validation_items = 0  # 2
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


101
597
16.917922948073702


In [ ]:
validation_items = 0 # cluster 1
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


50
899
5.561735261401557


In [ ]:
validation_items = 0   # 0
train_items = 0
for keys, values in train_val_split.items():
    validation_items += values[0]
    train_items += values[1]
print(validation_items)
print(train_items)
print(validation_items / train_items * 100)


103
818
12.591687041564793


In [5]:
datasets_npz.keys()

dict_keys(['rwc', 'gtzan', 'smc', 'guitarset', 'jaah', 'tapcorrect', 'harmonix', 'beatles', 'ballroom', 'simac', 'asap', 'hainsworth'])

In [24]:
np.savez("spectrograms.npz", spectrograms=spectrograms)

In [ ]:
dataset = "gtzan"
spectrograms = datasets_npz[dataset]
#spectrograms = {str(k): v for k, v in spectrograms.items()}
save_spectrograms_path = os.path.join(root_save, f"cluster_{cluster_number}/data/audio/spectrograms")
path_to_save = os.path.join(save_spectrograms_path , f"{dataset}.npz")
np.savez(path_to_save, **spectrograms)

In [4]:
import numpy as np
data = np.load("/hpcwork/ui556004/data/beat_this/clusters/cluster_1/data/audio/spectrograms/harmonix.npz")
lst = data.files
len(lst) / 22

487.0

In [47]:
dataset, remainder = lst[0].split("/", 1)
remainder

'track'

In [ ]:
import numpy as np
import os
from tqdm import tqdm
root = "/hpcwork/ui556004/data/beat_this/"
# filtering files that belong to cluster number 0
cluster = {}
df_filtered = df[df["label"].str.contains("0")]
files = list(df_filtered["file"].values)
# getting all files and corresponding datasets
files_wo_dataset = [file.split("___")[1] for file in files]
datasets_used = set([file.split("___")[0] for file in files])
print(len(df_filtered))
print(datasets_used)

gtzan_files = 0
cluster = {}
for dataset in tqdm(datasets_used):
#dataset = "asap"
    data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
    lst = data_npz.files
    npz_files_filtered = set([file.split("/")[0] for file in lst])
    selected_files = [file for file in npz_files_filtered if file in files_wo_dataset]
    print(f"number of selected files from the {dataset} is {len(selected_files)}")   
    for piece in selected_files:
        # getting all augmentations of the same file
        all_similar = [file for file in lst if piece in file ]
        if (len(all_similar) != 22 and len(all_similar) == 1):
            gtzan_files += 1
            #print(len(all_similar))
        
        pieces = {f"{dataset}___{file}" : data_npz[file] for file in all_similar}
        cluster = {**cluster, **pieces }
# sanity check: except for gtzan, each file should be repeated 22 times
assert (len(cluster) - gtzan_files) / 22 + gtzan_files == len(df_filtered)

In [19]:
import pandas as pd
split = pd.read_csv("/hpcwork/ui556004/data/beat_this/clusters/cluster_0/data/annotations/smc/single.split", sep = "\t", header = None, names = ["File", "Split"])
# split_filtered = split[split["Split"] != "train"]
# list(split_filtered["File"].values)
split

,File,Split
0,smc_001,train
1,smc_002,train
2,smc_003,train
3,smc_004,train
4,smc_005,train
...,...,...
212,smc_285,train
213,smc_286,train
214,smc_287,train
215,smc_288,train


In [1]:
import os 
checkpoint_path = "/hpcwork/ui556004/results/beat_this/checkpoints"
checkpoint_folder = os.path.join(checkpoint_path, "lalala")
os.makedirs(checkpoint_folder, exist_ok = True)

# test scores

In [3]:
import json
json_file = open("test_scores.json")
json1_str = json_file.read()
json1_data = json.loads(json1_str)
json1_data

{'gtzan/gtzan_blues_00000/track.npy': {'F-measure_beat': 0.9026548672566371,
  'Cemgil_beat': 0.7620232756812331,
  'CMLt_beat': 0.75,
  'AMLt_beat': 0.75,
  'F-measure_downbeat': 0.5652173913043478,
  'Cemgil_downbeat': 0.5736816673527043,
  'CMLt_downbeat': 0.03125,
  'AMLt_downbeat': 0.75},
 'gtzan/gtzan_blues_00001/track.npy': {'F-measure_beat': 0.8421052631578947,
  'Cemgil_beat': 0.6865430886973933,
  'CMLt_beat': 0.6206896551724138,
  'AMLt_beat': 0.6206896551724138,
  'F-measure_downbeat': 0.4666666666666667,
  'Cemgil_downbeat': 0.5071170481479714,
  'CMLt_downbeat': 0.0,
  'AMLt_downbeat': 0.30434782608695654},
 'gtzan/gtzan_blues_00002/track.npy': {'F-measure_beat': 0.8503937007874016,
  'Cemgil_beat': 0.7439014125976526,
  'CMLt_beat': 0.5846153846153846,
  'AMLt_beat': 0.5846153846153846,
  'F-measure_downbeat': 0.6666666666666667,
  'Cemgil_downbeat': 0.5691718758173158,
  'CMLt_downbeat': 0.5294117647058824,
  'AMLt_downbeat': 0.5294117647058824},
 'gtzan/gtzan_blues_000

In [4]:
import json
json_file = open("json_val_scores/clusters2_layer6/1/best-seed0-epoch=01-valfval_F-measure_beat=0.9725_orig.json")
json1_str = json_file.read()
json1_data = json.loads(json1_str)
json1_data

{'ballroom/ballroom_Albums-Cafe_Paradiso-01/track.npy': {'F-measure_beat': 1.0,
  'Cemgil_beat': 0.9167726728219816,
  'CMLt_beat': 1.0,
  'AMLt_beat': 1.0,
  'F-measure_downbeat': 1.0,
  'Cemgil_downbeat': 0.8717540391733888,
  'CMLt_downbeat': 1.0,
  'AMLt_downbeat': 1.0},
 'ballroom/ballroom_Albums-Cafe_Paradiso-06/track.npy': {'F-measure_beat': 1.0,
  'Cemgil_beat': 0.9221384804767336,
  'CMLt_beat': 1.0,
  'AMLt_beat': 1.0,
  'F-measure_downbeat': 1.0,
  'Cemgil_downbeat': 0.9378866858274131,
  'CMLt_downbeat': 1.0,
  'AMLt_downbeat': 1.0},
 'ballroom/ballroom_Albums-Cafe_Paradiso-07/track.npy': {'F-measure_beat': 1.0,
  'Cemgil_beat': 0.9564673263702477,
  'CMLt_beat': 1.0,
  'AMLt_beat': 1.0,
  'F-measure_downbeat': 1.0,
  'Cemgil_downbeat': 0.9952666514578993,
  'CMLt_downbeat': 1.0,
  'AMLt_downbeat': 1.0},
 'ballroom/ballroom_Albums-Cafe_Paradiso-12/track.npy': {'F-measure_beat': 0.9876543209876543,
  'Cemgil_beat': 0.9400440279570309,
  'CMLt_beat': 0.975609756097561,
  'AML

the val score can be a little differet from the one obtained during training because they only use middle of some pieces
Warning: for performances, this only runs on the middle excerpt of the long pieces

In [5]:
mean_f1 = 0    # sanity check that compute paper metrics just averages all f1 scores over tracks

for key, values in json1_data.items():
    mean_f1 += values['F-measure_beat']
mean_f1 / (len(json1_data))    # actual   


0.9664921376350432

In [7]:
json1_data["gtzan/gtzan_blues_00000/track.npy"]['F-measure_beat']

0.9026548672566371

In [ ]:
mean_f1 = 0    # sanity check that compute paper metrics just averages all f1 scores over tracks

for key, values in json1_data.items():
    mean_f1 += values['F-measure_beat']
mean_f1 / (len(json1_data))    # actual reported f1 acc was 0.7855876129845742

0.7855876129845732

In [33]:
import os
os.listdir("/hpcwork/ui556004/results/beat_this/checkpoints/0full_data_intermediate_checkpointsS1shift_tolerant_weighted_bce-h512-augTrueTrueTrue/best")

['best-epoch=00-valfval_F-measure_beat=0.3698.ckpt',
 'best-epoch=00-valfval_F-measure_beat=0.6354.ckpt']

In [5]:
from pathlib import Path
import os
best_ckpt_path= "rwthfs/rz/cluster/hpcwork/ui556004/results/beat_this/checkpoints/0full_data_intermediate_checkpointsS1shift_tolerant_weighted_bce-h512-augTrueTrueTrue/best/best-epoch=00-valfval_F-measure_beat=0.6619-v1.ckpt"
new_name = Path(best_ckpt_path).stem + "_orig.ckpt"
folder = os.path.dirname(best_ckpt_path)
new_path = os.path.join(folder, new_name )
new_path

'rwthfs/rz/cluster/hpcwork/ui556004/results/beat_this/checkpoints/0full_data_intermediate_checkpointsS1shift_tolerant_weighted_bce-h512-augTrueTrueTrue/best/best-epoch=00-valfval_F-measure_beat=0.6619-v1_orig.ckpt'

In [13]:
data = np.load("/hpcwork/ui556004/data/beat_this/clusters/cluster_0/data/audio/spectrograms/gtzan.npz")
lst = data.files
len(lst) 
lst[92]
spectrograms =data[lst[92]]
save_spectrograms_path = "/hpcwork/ui556004/data/beat_this/clusters/cluster_try/data/audio/spectrograms"
path_to_save = os.path.join(save_spectrograms_path , "gtzan.npz")
np.savez(path_to_save, **{lst[92]: spectrograms})

In [11]:
# sanity check for the data size 
size_wo_gtzan = 5556 -999
(size_wo_gtzan * 0.85 * 22) / 64

1331.4984375

## sanity check that the amount of data is okay

Training set oversampled from 4000 to 11960 excerpts.

In [17]:
11960 / 8   # 8 is a batch size

1495.0

Validation set: 556 

In [ ]:
556 / 8  

69.5

In [5]:
import numpy as np
import os
from tqdm import tqdm
import pandas as pd
import shutil
def get_split_files():
    root = "/hpcwork/ui556004/data/beat_this/"
    # save_spectrograms_path = os.path.join(root, f"data/audio/spectrograms")
    # annotations_path = os.path.join(root, f"data/annotations")
    #shutil.copytree("/hpcwork/ui556004/data/beat_this/clusters/annotations", annotations_path, dirs_exist_ok = True)
    #os.makedirs(os.path.join(root_save, f"cluster_{cluster_number}/data/annotations"), exist_ok = True)
    # filtering files that belong to cluster number 0
    #cluster = {}
    #df_filtered = df[df["label"].str.contains(str(cluster_number))]
    #files = list(df_filtered["file"].values)
    # getting all files and corresponding datasets
    #files_wo_dataset = [file.split("___")[1] for file in files]
    datasets_used =  os.listdir(root)#set([file.split("___")[0] for file in files])
    datasets_used = [file[:-4] for file in datasets_used if file.endswith("npz")]
    train_val_split = {}
    for dataset in tqdm(datasets_used):
        validation_files_new = 0
        train_files_new = 0
        data_npz = np.load(os.path.join(root, f"{dataset}.npz"))
        lst = data_npz.files
        npz_files_filtered = set([file.split("/")[0] for file in lst])
        selected_files = npz_files_filtered  #[file for file in npz_files_filtered if file in files_wo_dataset]
        #print(f"number of selected files from the {dataset} is {len(selected_files)}")   
        if dataset != "gtzan":
            split = pd.read_csv(f"/hpcwork/ui556004/data/beat_this/data/annotations/{dataset}/single.split", sep = "\t", header = None, names = ["File", "Split"])
            split_filtered = split[split["Split"] != "train"]
            val_files_split = list(split_filtered["File"].values)
            for piece in selected_files:
                if piece in val_files_split:
                    validation_files_new += 1
                else:
                    train_files_new += 1
            train_val_split[dataset] =  (validation_files_new, train_files_new)
        
    return train_val_split
    


In [9]:
train_val_split =  get_split_files()
train_val_split

100%|██████████| 16/16 [00:00<00:00, 17.78it/s]


{'beatles': (27, 153),
 'groove_midi': (51, 285),
 'hjdb': (35, 200),
 'guitarset': (27, 153),
 'harmonix': (137, 774),
 'rwc': (34, 192),
 'ballroom': (103, 582),
 'smc': (0, 217),
 'candombe': (5, 30),
 'jaah': (17, 96),
 'simac': (0, 595),
 'tapcorrect': (15, 86),
 'asap': (65, 408),
 'hainsworth': (33, 189),
 'filosax': (7, 41)}

In [13]:

sum_train = 0
sum_val = 0
for key, value in train_val_split.items():
    sum_train += value[1]
    sum_val += value[0]
print(f"Total of train items : {sum_train}, total of val items : {sum_val}, total items :{sum_train + sum_val + 999}")

Total of train items : 4001, total of val items : 556, total items :5556


In [45]:
s = "hpcwork/ui556004/data/beat_this/clustering_configurations/clusters4_layer12/cluster_2"
s.split(os.sep)[-2:]

['clusters4_layer12', 'cluster_2']

### validation files intersection for c-means

In [8]:
import json
import os
json_val_path = "json_val_scores/clusters4_layer12"
keys = {}
for i in range (1,5):
    folder = os.path.join(json_val_path, str(i))
    file_scores = [file for file in os.listdir(folder) if "best_checkpoint" in file][0]

    json_file = open(os.path.join(folder, file_scores))
    json1_str = json_file.read()
    json1_data = json.loads(json1_str)
    clsuter_keys = set(json1_data.keys())
    keys[i] = clsuter_keys
print(keys)

{1: {'groove_midi/drummer7_session3_92_hiphop_70_beat_4-4/track.npy', 'harmonix/0553_alivefeatthegoodnatured/track.npy', 'groove_midi/drummer7_session3_106_hiphop_70_beat_4-4/track.npy', 'hjdb/hjdb_Future_Funk/track.npy', 'ballroom/ballroom_Media-103314/track.npy', 'harmonix/0573_babyludacris/track.npy', 'hjdb/hjdb_Spiritual_Aura/track.npy', 'harmonix/0967_walkingonadream/track.npy', 'harmonix/0432_letsgo/track.npy', 'hjdb/hjdb_Fires_Burning/track.npy', 'groove_midi/drummer7_session1_14_jazz_100_beat_4-4/track.npy', 'harmonix/0745_immabewolf/track.npy', 'hjdb/hjdb_Jump_Mk_II/track.npy', 'groove_midi/drummer9_session1_1_rock_100_beat_4-4/track.npy', 'groove_midi/drummer10_session1_3_jazz-swing_215_beat_4-4/track.npy', 'hjdb/hjdb_Darkman/track.npy', 'ballroom/ballroom_Albums-Cafe_Paradiso-07/track.npy', 'ballroom/ballroom_Media-106103/track.npy', 'hjdb/hjdb_Unfriendly/track.npy', 'groove_midi/drummer3_session2_6_rock_100_beat_4-4/track.npy', 'hainsworth/hainsworth_172/track.npy', 'harmon

finding out how many validation files are intersecting with other clusters

In [11]:
for i in range(1,5):
    for j in range(i+1,5):
        intersection = keys[i].intersection(keys[j])
        print(f"intersection between cluster {i} and cluster {j} has {len(intersection)} items")

intersection between cluster 1 and cluster 2 has 0 items
intersection between cluster 1 and cluster 3 has 28 items
intersection between cluster 1 and cluster 4 has 0 items
intersection between cluster 2 and cluster 3 has 18 items
intersection between cluster 2 and cluster 4 has 0 items
intersection between cluster 3 and cluster 4 has 0 items


In [20]:
def findTaskPairForSlot(taskDurations, slotLength):
    # Write your code here
    n = len(taskDurations)
    for i in range (n):
        for j in range (i+1, n):
            if taskDurations[i] + taskDurations[j] == slotLength:
                return [i, j]
    return [-1, -1]

In [25]:
taskDurations = [2, 7, 2, 11, 15]
slotLength = 9
findTaskPairForSlot(taskDurations, slotLength)

[0, 1]

# segfault debug

In [3]:
pip install pytorch-lightning


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 849 kB 13.4 MB/s            
     |████████████████████████████████| 983 kB 136.5 MB/s            
     |████████████████████████████████| 1.7 MB 188.7 MB/s            
     |████████████████████████████████| 134 kB 165.5 MB/s            
     |████████████████████████████████| 67 kB 16.7 MB/s             
     |████████████████████████████████| 197 kB 139.7 MB/s            
     |████████████████████████████████| 219 kB 219.0 MB/s            
     |████████████████████████████████| 239 kB 176.1 MB/s            
     |████████████████████████████████| 346 kB 229.0 MB/s            
Note: you may need to restart the kernel to use updated packages.


In [8]:
from beat_this.dataset.mmnpz import MemmappedNpzFile
from pathlib import Path

for dataset in ["asap", "ballroom", "beatles", "candombe", "filosax", "groove_midi", "guitarset", "hainsworth", "harmonix", "hjdb", "jaah", "rwc", "tapcorrect"]:
    npz_file = Path(f"/hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/cluster_2/data/audio/spectrograms/{dataset}.npz")
    if npz_file.exists():
        try:
            print(f"Checking {npz_file}...")
            mm = MemmappedNpzFile(npz_file)
            # Try to access all keys
            for key in mm.keys():
                arr = mm[key]
                _ = arr.shape  # Force access
            print(f"  ✓ {npz_file} OK")
        except Exception as e:
            print(f"  ✗ {npz_file} CORRUPTED: {e}")

Checking /hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/cluster_2/data/audio/spectrograms/asap.npz...
  ✓ /hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/cluster_2/data/audio/spectrograms/asap.npz OK
Checking /hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/cluster_2/data/audio/spectrograms/ballroom.npz...
  ✓ /hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/cluster_2/data/audio/spectrograms/ballroom.npz OK
Checking /hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/cluster_2/data/audio/spectrograms/beatles.npz...
  ✓ /hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/cluster_2/data/audio/spectrograms/beatles.npz OK
Checking /hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/cluster_2/data/audio/spectrograms/filosax.npz...
  ✓ /hpcwork/ui556004/data/beat_this/clustering_configurations/clusters2_layer6/c

## extracting npz to the folder


In [5]:
import numpy as np
import os
from tqdm import tqdm
files = ["ballroom", "beatles", "filosax", "guitarset", "hainsworth", "harmonix", "hjdb", "jaah", "rwc", "simac", "smc", "tapcorrect"]
for file in tqdm(files): 
    print(f"processing file {file}")
    path = f"/hpcwork/ui556004/data/beat_this/{file}.npz"
    out_dir = f"/hpcwork/ui556004/data/beat_this/extracted/{file}"
    os.makedirs(out_dir, exist_ok=True)

    data = np.load(path)

    for name in data.files:
        # Build output path respecting subfolders in the key name
        out_path = os.path.join(out_dir, f"{name}.npy")
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        np.save(out_path, data[name])

    print("Done!")

  0%|          | 0/12 [00:00<?, ?it/s]

processing file ballroom


  8%|▊         | 1/12 [00:34<06:17, 34.35s/it]

Done!
processing file beatles


 17%|█▋        | 2/12 [00:48<03:46, 22.70s/it]

...

Done!
processing file smc


 92%|█████████▏| 11/12 [05:39<00:27, 27.85s/it]

Done!
processing file tapcorrect


100%|██████████| 12/12 [06:21<00:00, 31.77s/it]

Done!


In [ ]:
base_path = '/hpcwork/ui556004/data/beat_this/extracted'

'Bach_Fugue_bwv_846_Shi05M/track'